# pandas 之九：数据导入全攻略

---

真实工作里，数据可能来自 **Excel / CSV / TXT / JSON / 数据库**。
这篇把每种格式的读法讲一遍。

## 学习方法说明（重要）

> **原稿的问题**：原稿里读的 `student.xlsx`、`data.txt`、`file.xlsx`、
> `sales_data.xlsx`、`data.json`、`server.log` **全都不存在于项目里**，
> 而且路径还写错了（`.生成文件` 少了个斜杠）。
> 结果就是：这份笔记**一行都跑不通**，只能当"参数说明书"看。
>
> **本篇的改进**：**先用代码把演示文件造出来，再读它们。**
> 这样每一段都是可运行的，你能亲眼看到每个参数的效果。
> 演示文件都放在**系统临时目录**里，不会污染你的项目文件夹。

## 学习目标

1. 掌握 `read_excel` 的分表、表头、列选择
2. 掌握 `read_csv` 的分隔符、编码、缺失值
3. 会读 TXT（制表符 / 空格 / 固定宽度）
4. 会读 JSON（普通 / JSON Lines / 嵌套）


In [ ]:
import pandas as pd
import numpy as np
import os
import tempfile

print('pandas 版本：', pd.__version__)

# 建一个临时目录放演示文件
DEMO = tempfile.mkdtemp(prefix='import_demo_')
print('\n演示文件目录：', DEMO)


---
# 第一部分：生成演示文件

我们造一份"学生成绩"数据，做成各种格式：


In [ ]:
# 基础数据
students = pd.DataFrame({
    'ID': [1001, 1002, 1003, 1004, 1005],
    '姓名': ['张三', '李四', '王五', '赵六', '钱七'],
    '班级': ['一班', '一班', '二班', '二班', '三班'],
    '数学': [85, 92, 78, 95, 88],
    '英语': [76, 88, 90, 82, 91],
    '语文': [90, 85, 88, 79, 86],
})
print(students)


### 生成 Excel（多工作表）


In [ ]:
excel_path = os.path.join(DEMO, 'student.xlsx')

with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    students.to_excel(writer, sheet_name='成绩表', index=False)
    # 第二个 sheet：只有汇总信息
    students[['ID', '姓名']].to_excel(writer, sheet_name='名单', index=False)
    # 第三个 sheet：带标题行的（表头不在第一行）
    students.to_excel(writer, sheet_name='带标题', index=False, startrow=1)

print('已生成 student.xlsx，包含 3 个 sheet：成绩表 / 名单 / 带标题')


### 生成 CSV 和 TXT


In [ ]:
# 1) 普通 CSV（utf-8-sig，中文 Windows 友好）
csv_path = os.path.join(DEMO, 'student.csv')
students.to_csv(csv_path, index=False, encoding='utf-8-sig')

# 2) 制表符分隔的 TXT
tsv_path = os.path.join(DEMO, 'student_tab.txt')
students.to_csv(tsv_path, sep='\t', index=False, encoding='utf-8')

# 3) 空格分隔的 TXT（多空格）
space_path = os.path.join(DEMO, 'student_space.txt')
# 用【两个空格】分隔，模拟老系统导出格式
# 注意：to_csv 的 sep 只能是【单个字符】，所以这里手工把文件写出来
with open(space_path, 'w', encoding='utf-8') as f:
    f.write('  '.join(students.columns) + '\n')
    for _, row in students.iterrows():
        f.write('  '.join(str(v) for v in row.values) + '\n')

print('已生成：student.csv / student_tab.txt / student_space.txt')
print('\nstudent_space.txt 的实际内容（列之间是两个空格）：')
with open(space_path, encoding='utf-8') as f:
    print(f.read())


In [ ]:
# 4) 带注释行的 TXT
comment_path = os.path.join(DEMO, 'with_comment.txt')
with open(comment_path, 'w', encoding='utf-8') as f:
    f.write('# 这是数据说明，应该被跳过\n')
    f.write('# 生成时间：2026\n')
    students.to_csv(f, sep='\t', index=False)
print('已生成 with_comment.txt（前两行是注释）')

# 5) 固定宽度 TXT
fwf_path = os.path.join(DEMO, 'fixed_width.txt')
with open(fwf_path, 'w', encoding='utf-8') as f:
    f.write('ID    姓名    分数\n')
    f.write('1001  张三    85\n')
    f.write('1002  李四    92\n')
    f.write('1003  王五    78\n')
print('已生成 fixed_width.txt（列是"对齐"的，没有分隔符）')


### 生成 JSON 的三种形态


In [ ]:
import json

# 1) 普通 JSON（列式：key 是列名）
records = students.to_dict(orient='list')          # {'ID': [...], '姓名': [...]}
json_col_path = os.path.join(DEMO, 'data_col.json')
with open(json_col_path, 'w', encoding='utf-8') as f:
    json.dump(records, f, ensure_ascii=False)
print('1) data_col.json（列式）')

# 2) 记录式 JSON
json_rec_path = os.path.join(DEMO, 'data_rec.json')
with open(json_rec_path, 'w', encoding='utf-8') as f:
    json.dump(students.to_dict(orient='records'), f, ensure_ascii=False)
print('2) data_rec.json（记录式）')

# 3) JSON Lines（每行一个独立 JSON 对象）
jsonl_path = os.path.join(DEMO, 'data.jsonl')
with open(jsonl_path, 'w', encoding='utf-8') as f:
    for rec in students.to_dict(orient='records'):
        f.write(json.dumps(rec, ensure_ascii=False) + '\n')
print('3) data.jsonl（每行一个对象）')


### 生成嵌套 JSON


In [ ]:
nested = {
    'users': [
        {
            'name': 'John',
            'age': 30,
            'address': {
                'street': '123 Main St',
                'city': 'New York',
                'coordinates': {'lat': 40.7128, 'lng': -74.0060},
            },
        },
        {
            'name': 'Alice',
            'age': 25,
            'address': {
                'street': '456 Oak Ave',
                'city': 'London',
                'coordinates': {'lat': 51.5074, 'lng': -0.1278},
            },
        },
    ]
}
nested_path = os.path.join(DEMO, 'nested.json')
with open(nested_path, 'w', encoding='utf-8') as f:
    json.dump(nested, f, ensure_ascii=False, indent=2)
print('4) nested.json（嵌套结构）')

print('\n所有演示文件已就绪：')
for fn in sorted(os.listdir(DEMO)):
    print(' ', fn)


---
# 第二部分：读 Excel

## 一、`read_excel` 常用参数

| 参数 | 作用 |
|---|---|
| `sheet_name` | 读哪个工作表（名字 / 索引 / 列表 / `None`） |
| `header` | 哪一行是列名 |
| `usecols` | 读哪几列 |
| `nrows` | 读前 N 行 |
| `dtype` | 指定列类型 |
| `index_col` | 哪列当索引 |
| `na_values` | 哪些值算缺失 |


In [ ]:
# 默认读第一个 sheet
df_default = pd.read_excel(excel_path)
print('默认读第 1 个 sheet（成绩表）：')
print(df_default)


### `sheet_name` 的 4 种用法


In [ ]:
print("1) 按【名称】读：")
print(pd.read_excel(excel_path, sheet_name='名单'))

print("\n2) 按【索引】读（从 0 开始，1 = 第二个 sheet）：")
print(pd.read_excel(excel_path, sheet_name=1))


In [ ]:
print("3) sheet_name=None -> 一次读【所有】sheet，返回字典：")
all_sheets = pd.read_excel(excel_path, sheet_name=None)
print('类型：', type(all_sheets).__name__)
print('包含的 sheet：', list(all_sheets.keys()))
print('\n取出"名单"这个 sheet：')
print(all_sheets['名单'])


> **`sheet_name=None` 是很实用的一招**：
> 当你不知道 Excel 里有哪些 sheet 时，一把全读进来，
> 再用 `.keys()` 看结构。**处理别人给的 Excel 时特别有用。**


In [ ]:
print("4) 传列表，只读指定的几个：")
two = pd.read_excel(excel_path, sheet_name=['成绩表', '名单'])
print('拿到', len(two), '个 sheet：', list(two.keys()))


### `header`：指定表头在哪一行

有些 Excel 第一行是大标题，真正的列名在第二行。


In [ ]:
print('原始"带标题"sheet（第一行是标题）：')
print(pd.read_excel(excel_path, sheet_name='带标题', header=None).head(3))


**`header=1` 表示"用第 2 行（0 开始计数）当列名"**：


In [ ]:
print('用 header=1 指定第二行当列名：')
print(pd.read_excel(excel_path, sheet_name='带标题', header=1))


**`header=None` 表示"这个文件没有列名"**，pandas 自动用 `0,1,2...` 当列名：


In [ ]:
print('header=None（不要表头）：')
print(pd.read_excel(excel_path, sheet_name='成绩表', header=None).head(3))


### `usecols`：只读需要的列（大文件时很省内存）


In [ ]:
print("1) 按列名选：usecols=['ID','姓名','数学']")
print(pd.read_excel(excel_path, usecols=['ID', '姓名', '数学']))

print("\n2) 按位置选：usecols=[0, 1, 3]")
print(pd.read_excel(excel_path, usecols=[0, 1, 3]))

print("\n3) 按字母区间：usecols='A:C'")
print(pd.read_excel(excel_path, usecols='A:C'))


### `nrows` / `index_col` / `dtype`


In [ ]:
print('nrows=3 只读前 3 行（快速预览大文件）：')
print(pd.read_excel(excel_path, nrows=3))

print('\nindex_col="ID" 把 ID 列当索引：')
print(pd.read_excel(excel_path, index_col='ID'))


In [ ]:
print('dtype 指定列类型（比如把 ID 读成字符串，避免丢失前导 0）：')
d = pd.read_excel(excel_path, dtype={'ID': str})
print(d.dtypes)
print('\nID 现在是字符串：', d['ID'].tolist())


> **为什么有时候要把数字列读成字符串？**
> 像**工号、身份证号、手机号、邮编**这类"编号型"数字，
> 前导 0 很重要（`001234` vs `1234`），
> 而且它们不参与运算。**这类列应该读成字符串。**


---
# 第三部分：读 CSV / TXT

## 二、`read_csv` 其实什么文本都能读

**`read_csv` 不只读 `.csv`** —— 它读的是"结构化文本"，
换个分隔符就能读 TXT。


In [ ]:
print('=== 1) 标准 CSV ===')
print(pd.read_csv(csv_path, encoding='utf-8-sig').head(3))

print('\n=== 2) 制表符分隔的 TXT ===')
print(pd.read_csv(tsv_path, sep='\t').head(3))


In [ ]:
print('=== 3) 多个空格分隔 ===')
print(pd.read_csv(space_path, sep=r'\s+').head(3))


**反例：如果用 `sep=' '`（单个空格）读这个文件会怎样？**


In [ ]:
bad = pd.read_csv(space_path, sep=' ')
print("用 sep=' ' 读出来的列名：")
print(list(bad.columns))
print('\n前两行（注意多了很多空列）：')
print(bad.head(2))
print("\n  <- 因为两个空格之间被当成了'一个空字段'，于是多出一堆 NaN 列")
print("  <- 正确写法是 sep=r'\\s+'，把'一个或多个空白'看成一个分隔符")


> **`sep=r'\s+'` 的含义**：正则表达式，匹配"一个或多个空白字符"。
> 所以不管是 1 个空格、2 个空格还是 Tab，都能正确切分。
> **`r` 前缀表示"原始字符串"**，这样 `\s` 不会被 Python 先转义一遍。

> ⚠️ **版本提醒**：老教程里常写 `delim_whitespace=True`，
> 这个参数**在 pandas 3.0 已被删除**，必须改用 `sep=r'\s+'`。


In [ ]:
try:
    pd.read_csv(space_path, delim_whitespace=True)
except TypeError as e:
    print('delim_whitespace 已删除 -> 报错：')
    print(str(e)[:120])


### `comment`：跳过注释行


In [ ]:
print('原始文件内容：')
with open(comment_path, encoding='utf-8') as f:
    print(f.read())

print('用 comment="#" 跳过注释行：')
print(pd.read_csv(comment_path, sep='\t', comment='#'))


### 固定宽度文件：`read_fwf`

有些老系统导出的 TXT **没有分隔符**，靠"列对齐"来区分。
这种要用 `read_fwf`（Fixed Width File）。


In [ ]:
print('原始文件（靠空格对齐）：')
with open(fwf_path, encoding='utf-8') as f:
    print(f.read())

print('用 widths 指定每列宽度：')
print(pd.read_fwf(fwf_path, widths=[6, 8, 8]))


### `encoding`：中文乱码的终极解决方案

| 场景 | 推荐编码 |
|---|---|
| Windows Excel 导出的 CSV（带 BOM） | **`utf-8-sig`** |
| 老系统 / 国内软件导出 | **`gbk`** |
| 标准 UTF-8 | `utf-8` |

**排查顺序：先试 `utf-8-sig`，再试 `gbk`。**


In [ ]:
print('同一个文件用不同编码读的结果差异：')
print('utf-8-sig :', pd.read_csv(csv_path, encoding='utf-8-sig').columns.tolist())

try:
    cols = pd.read_csv(csv_path, encoding='utf-8').columns.tolist()
    print('utf-8     :', cols)
except Exception as e:
    print('utf-8     :', str(e)[:80])


---
# 第四部分：读 JSON

## 三、JSON 的三种形态

| 形态 | 长相 | 读法 |
|---|---|---|
| **列式** | `{"name":["A","B"], "age":[1,2]}` | `read_json()` 直接读 |
| **记录式** | `[{"name":"A"},{"name":"B"}]` | `read_json()` 直接读 |
| **JSON Lines** | 每行一个独立对象 | **`read_json(..., lines=True)`** |


In [ ]:
print('=== 1) 列式 JSON ===')
print(pd.read_json(json_col_path))

print('\n=== 2) 记录式 JSON ===')
print(pd.read_json(json_rec_path))

print('\n=== 3) JSON Lines（关键参数 lines=True）===')
print(pd.read_json(jsonl_path, lines=True))


### ⚠️ 读 JSON Lines 忘了 `lines=True` 会怎样？


In [ ]:
try:
    pd.read_json(jsonl_path)
except ValueError as e:
    print('不加 lines=True -> 报错：')
    print(str(e)[:200])


### 从 JSON **字符串**读取：要包一层 `StringIO`

⚠️ **pandas 3.0 的行为变化**：`pd.read_json('{...}')` 会把这段字符串
**当成文件路径**，然后报 `FileNotFoundError`。

**正确做法：用 `io.StringIO` 把它包装成一个"文件对象"。**


In [ ]:
import io

single = '{"name": "John", "age": 30, "city": "New York"}'
print('要读的 JSON 字符串：', single)

# 直接传字符串 -> 被当成路径，报错
try:
    pd.read_json(single)
except FileNotFoundError as e:
    print('\n直接传字符串 -> 报错：')
    print('  ', str(e)[:80])


**正确写法**：`io.StringIO` 把字符串变成"像文件一样"的对象。


In [ ]:
print('用 io.StringIO 包装后，读成 Series（单个 JSON 对象要用 typ="series"）：')
print(pd.read_json(io.StringIO(single), typ='series'))


In [ ]:
# 字符串形式的"数组"，用 StringIO 也能读成 DataFrame
arr_str = '[{"name":"A","age":1},{"name":"B","age":2}]'
print('数组形式的 JSON 字符串：')
print(pd.read_json(io.StringIO(arr_str)))


### 嵌套 JSON：`json_normalize`

这是处理嵌套 JSON 的**主力工具**——把"层级结构"压平成表格，
**嵌套的层用 `.` 连接成列名**。


In [ ]:
print('原始嵌套 JSON：')
print(json.dumps(nested, ensure_ascii=False, indent=2)[:400], '...')


In [ ]:
flat = pd.json_normalize(nested['users'])
print('\njson_normalize 压平后：')
print(flat)
print('\n列名：')
print(list(flat.columns))
print('  <- 嵌套的 address 被展开成 address.street / address.city / ...')


**压平逻辑**：

```
原始:                    压平后:
{                        ┌──────────────┬──────────────┬─────────────────────┐
  "name": "John",        │ name │  age  │ address.city │ address.coordinates.lat │
  "address": {           ├──────┼───────┼──────────────┼─────────────────────────┤
    "city": "New York",  │ John │  30   │  New York    │  40.7128                │
    "coordinates": {...} │ Alice│  25   │  London      │  51.5074                │
  }                      └──────┴───────┴──────────────┴─────────────────────────┘
}
```


In [ ]:
# sep 参数可以自定义连接符（默认是 "."）
flat2 = pd.json_normalize(nested['users'], sep='_')
print('用 sep="_" 之后的列名：')
print(list(flat2.columns))


---
## 四、各格式读写对照表

| 格式 | 读 | 写 |
|---|---|---|
| CSV | `pd.read_csv(f)` | `df.to_csv(f, index=False)` |
| Excel | `pd.read_excel(f, sheet_name=...)` | `df.to_excel(f)` |
| TXT（分隔） | `pd.read_csv(f, sep=...)` | `df.to_csv(f, sep='\t')` |
| TXT（定宽） | `pd.read_fwf(f, widths=...)` | — |
| JSON | `pd.read_json(f)` | `df.to_json(f)` |
| JSON Lines | `pd.read_json(f, lines=True)` | `df.to_json(f, lines=True)` |
| 嵌套 JSON | `pd.json_normalize(data)` | — |
| SQL | `pd.read_sql(query, conn)` | `df.to_sql(name, conn)` |

---

## 五、常见坑清单

| 坑 | 现象 | 正解 |
|---|---|---|
| 读中文 CSV 报编码错 | 读不进 | 先试 `utf-8-sig`，再试 `gbk` |
| 用 `delim_whitespace=True` | pandas 3.0 报错 | 改用 `sep=r'\s+'` |
| 多空格文件用 `sep=' '` | 多出很多空列 | 用 `sep=r'\s+'` |
| JSON Lines 忘 `lines=True` | `ValueError` | 加上这个参数 |
| 单条 JSON 直接读 | 报错 | `typ='series'` |
| 传 JSON **字符串**给 `read_json` | `FileNotFoundError`（被当路径） | 用 `io.StringIO(...)` 包一层 |
| 嵌套 JSON 直接读 | 列里还是字典 | 用 `json_normalize` |
| 编号列被读成数字 | 前导 0 丢失 | `dtype={'ID': str}` |
| 表头不在第一行 | 列名变成 0,1,2 | 用 `header=n` |
| `usecols=['A','C','E']` 当字母区间 | 报错 | 区间用单个字符串 `'A:C'` |
| 不知道 Excel 有几个 sheet | 一个个试 | `sheet_name=None` 全读 |

---

## 六、动手练习

**练习 1**：把上面的 `students` 存成 CSV（`utf-8-sig`）和 Excel，再分别读回来，确认内容一致。

**练习 2**：读 `student.xlsx` 的"名单"sheet，并用 `sheet_name=None` 一次读全部 sheet。

**练习 3**：读 `student_tab.txt`（制表符）和 `student_space.txt`（多空格），说清各自用什么 `sep`。

**练习 4**：读 `with_comment.txt` 时跳过以 `#` 开头的行。

**练习 5**：读 `data.jsonl`，说明为什么必须加 `lines=True`。

**练习 6**：用 `json_normalize` 把 `nested.json` 压平，并说出生成的新列名。

**练习 7**：读 `student.xlsx` 时只读 `ID`、`姓名`、`数学` 三列，并把 `ID` 读成字符串。

---

**答案思路（先自己做）**

```python
# 1
students.to_csv(os.path.join(DEMO, 'a.csv'), index=False, encoding='utf-8-sig')
print(pd.read_csv(os.path.join(DEMO, 'a.csv'), encoding='utf-8-sig').equals(students))

# 2
print(pd.read_excel(excel_path, sheet_name='名单'))
print(list(pd.read_excel(excel_path, sheet_name=None).keys()))

# 3
pd.read_csv(tsv_path, sep='\t')
pd.read_csv(space_path, sep=r'\s+')

# 4
pd.read_csv(comment_path, sep='\t', comment='#')

# 5
pd.read_json(jsonl_path, lines=True)     # 每行是独立对象，必须按行解析

# 6
flat = pd.json_normalize(nested['users'])
print(list(flat.columns))

# 7
pd.read_excel(excel_path, usecols=['ID', '姓名', '数学'], dtype={'ID': str})
```
